# Ética en la Inteligencia Artificial
## Módulo 5 — Notebook de Resumen

Este notebook presenta un recorrido por los conceptos técnicos clave en ética de IA:
1. Métricas de equidad y detección de sesgo
2. Fundamentos de explicabilidad
3. Privacidad diferencial
4. Simulación de impacto social

Ejecuta cada celda para ver las demostraciones.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

np.random.seed(42)
print("All imports successful.")

All imports successful.


## 1. Métricas de Equidad

Generamos un dataset sintético y calculamos métricas de equidad estándar entre grupos demográficos.

In [2]:
def generate_dataset(n=2000):
    race = np.random.choice(['A', 'B'], size=n, p=[0.7, 0.3])
    income = np.where(race == 'A',
                      np.random.normal(70, 20, n),
                      np.random.normal(50, 15, n))
    score = np.where(race == 'A',
                     np.random.normal(700, 50, n),
                     np.random.normal(650, 60, n))
    default = np.random.binomial(1, 0.1 + 0.3 * (score < 650).astype(int))
    return pd.DataFrame({'race': race, 'income': income, 'score': score, 'default': default})

df = generate_dataset()
print("Base rate by group:")
print(df.groupby('race')['default'].mean().round(3))

Base rate by group:
race
A    0.144
B    0.272
Name: default, dtype: float64


In [3]:
def fairness_audit(y_true, y_pred, protected):
    groups = np.unique(protected)
    for g in groups:
        mask = protected == g
        yt, yp = y_true[mask], y_pred[mask]
        tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        approval = yp.mean()
        print(f"Group {g}: TPR={tpr:.3f}, FPR={fpr:.3f}, Approval={approval:.3f}")

X = df[['income', 'score']]
y = df['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

protected = df.iloc[X_test.index]['race'].values
print("Fairness Audit:")
fairness_audit(y_test.values, y_pred, protected)

diff = (y_pred[protected == 'A'].mean() - y_pred[protected == 'B'].mean())
print(f"\nDemographic Parity Difference: {diff:.4f}")

Fairness Audit:
Group A: TPR=0.000, FPR=0.003, Approval=0.002
Group B: TPR=0.093, FPR=0.047, Approval=0.060

Demographic Parity Difference: -0.0577


## 2. Explicabilidad

Inspeccionamos los coeficientes del modelo para un modelo interpretable.

In [4]:
coef_df = pd.DataFrame({
    'feature': ['income', 'score'],
    'coefficient': model.coef_[0]
})
print("Model Coefficients (interpretable explanation):")
print(coef_df)
print("\nPositive coefficient = increases default probability.")

Model Coefficients (interpretable explanation):
  feature  coefficient
0  income    -0.000209
1   score    -0.013085

Positive coefficient = increases default probability.


## 3. Privacidad Diferencial

Implementamos el mecanismo de Laplace para publicar la media de un dataset con garantías formales de privacidad.

In [5]:
def laplace_mechanism(true_value, sensitivity, epsilon):
    noise = np.random.laplace(0, sensitivity / epsilon)
    return true_value + noise

data = np.random.normal(50, 15, 1000)
true_mean = np.mean(data)

epsilons = [0.01, 0.1, 1, 10]
print(f"True mean: {true_mean:.2f}")
for eps in epsilons:
    private = laplace_mechanism(true_mean, (data.max() - data.min()) / len(data), eps)
    print(f"  epsilon={eps:.2f}: private mean={private:.2f} (error={abs(private-true_mean):.2f})")

True mean: 50.18
  epsilon=0.01: private mean=64.89 (error=14.71)
  epsilon=0.10: private mean=49.69 (error=0.49)
  epsilon=1.00: private mean=50.23 (error=0.05)
  epsilon=10.00: private mean=50.19 (error=0.01)


## 4. Impacto Social: Comparación de Huella de Carbono

In [6]:
models = ['LogReg', 'Random Forest', 'BERT', 'GPT-3']
co2_estimates = [0.001, 0.01, 1.4, 500]
labels = [f'{v} tons' if v > 0.1 else f'{v*1000} kg' for v in co2_estimates]

fig = go.Figure()
fig.add_trace(go.Bar(
    y=models, x=co2_estimates, orientation='h',
    marker=dict(color=['#4C72B0', '#DD8452', '#55A868', '#C44E52']),
    text=labels, textposition='outside',
    hovertemplate='%{y}: %{x} tons<extra></extra>'
))
fig.update_layout(
    title='Huella de Carbono del Entrenamiento de Modelos de ML',
    xaxis=dict(title='CO₂ estimado (toneladas)', type='log'),
    yaxis=dict(title=''),
    height=400, width=700
)
fig.show()

## Resumen

Este notebook demostró cuatro áreas técnicas clave de la ética en IA:
1. **Métricas de equidad**: Detección de disparidades demográficas en predicciones de modelos
2. **Explicabilidad**: Comprender qué features impulsan las decisiones del modelo
3. **Privacidad diferencial**: Protección de datos individuales en estadísticas agregadas
4. **Impacto ambiental**: El costo de carbono de diferentes modelos de ML

Cada una de estas áreas se explora en profundidad en los notebooks de las lecciones.